In [ ]:
import pandas as pd
import sqlite3
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier


# Create database connection

In [ ]:
conn = sqlite3.connect("../data/olympics.db")
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
tables

athlethes = pd.read_sql("Select * from athletes", conn)
noc = pd.read_sql("select * from noc", conn)

In [ ]:
# Merge tables
df = pd.merge(athlethes, noc, on="NOC", how="left")
df.head()
df.info()
df.columns

<class 'pandas.DataFrame'>
RangeIndex: 48564 entries, 0 to 48563
Data columns (total 17 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   ID      48564 non-null  int64
 1   Name    48564 non-null  str  
 2   Sex     48564 non-null  str  
 3   Age     48564 non-null  int64
 4   Height  48564 non-null  int64
 5   Weight  48564 non-null  int64
 6   Team    48564 non-null  str  
 7   NOC     48564 non-null  str  
 8   Games   48564 non-null  str  
 9   Year    48564 non-null  int64
 10  Season  48564 non-null  str  
 11  City    48564 non-null  str  
 12  Sport   48564 non-null  str  
 13  Event   48564 non-null  str  
 14  Medal   48564 non-null  str  
 15  region  48564 non-null  str  
 16  notes   638 non-null    str  
dtypes: int64(5), str(12)
memory usage: 6.3 MB


Index(['ID', 'Name', 'Sex', 'Age', 'Height', 'Weight', 'Team', 'NOC', 'Games',
       'Year', 'Season', 'City', 'Sport', 'Event', 'Medal', 'region', 'notes'],
      dtype='str')

# Train and test dataset

In [ ]:

df['Medal'] = df['Medal'].apply(lambda x: 1 if x != 'None' else 0)
features =  ['Age', 'Event', 'Height', 'Sex', 'Sport', 'Weight', 'region']

target = ["Medal"]
x = df[features]
y = df[target]

x = pd.get_dummies(x, columns=['Event', 'region', 'Sex', 'Sport'])

x_train, x_test, y_train, y_test, = train_test_split(x, y, test_size=0.2, random_state=42)
x_train.head()


,Age,Height,Weight,Event_Alpine Skiing Men's Combined,Event_Alpine Skiing Men's Downhill,Event_Alpine Skiing Men's Giant Slalom,Event_Alpine Skiing Men's Slalom,Event_Alpine Skiing Men's Super G,Event_Alpine Skiing Women's Combined,Event_Alpine Skiing Women's Downhill,...,Sport_Freestyle Skiing,Sport_Ice Hockey,Sport_Luge,Sport_Military Ski Patrol,Sport_Nordic Combined,Sport_Short Track Speed Skating,Sport_Skeleton,Sport_Ski Jumping,Sport_Snowboarding,Sport_Speed Skating
1174,29,184,84,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
7901,26,180,65,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
27507,20,167,58,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
19973,33,176,60,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
15083,23,175,73,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False


In [12]:
y_train

,Medal
1174,0
7901,0
27507,0
19973,0
15083,1
...,...
11284,0
44732,0
38158,0
860,1


# Random Forest

In [ ]:
# Using smote for class imbalance
smote = SMOTE(random_state=42)
# Random Forest
x_train_smote, y_train_smote = smote.fit_resample(x_train, y_train.values.ravel()) 
rf = RandomForestClassifier(n_estimators=100, criterion='entropy')
# Training
rf.fit(x_train_smote, y_train_smote)
# Test
y_pred = rf.predict(x_test)
# Score
score = accuracy_score(y_test, y_pred)
score

0.8927210954391023

In [14]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.95      0.94      8595
           1       0.54      0.45      0.49      1118

    accuracy                           0.89      9713
   macro avg       0.74      0.70      0.72      9713
weighted avg       0.89      0.89      0.89      9713



In [15]:
y_train_smote

array([0, 0, 0, ..., 1, 1, 1], shape=(68548,))

# XGBoost

In [ ]:
# XGBoost
xgb = XGBClassifier(n_estimators=400)
# Training
xgb.fit(x_train_smote, y_train_smote)
# Test
y_pred = xgb.predict(x_test)
# Score
score = accuracy_score(y_test, y_pred)
score

0.8894265417481726

In [31]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.94      0.94      8595
           1       0.52      0.51      0.51      1118

    accuracy                           0.89      9713
   macro avg       0.73      0.72      0.73      9713
weighted avg       0.89      0.89      0.89      9713



In [51]:
# Hyperparametertuning Random Forest
param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],       # min Samples pro Blatt
    'max_features': ['sqrt', 'log2'],     # wie viele Features pro Split
    'bootstrap': [True, False]            # mit oder ohne Zurücklegen
}

rf_ft = RandomizedSearchCV(rf, param_distributions=param_grid, n_iter=20, cv=5, random_state=42)
rf_ft.fit(x_train_smote, y_train_smote)
rf_ft.best_params_

{'n_estimators': 100,
 'min_samples_split': 5,
 'min_samples_leaf': 1,
 'max_features': 'log2',
 'max_depth': None,
 'bootstrap': True}

In [53]:
y_pred = rf_ft.predict(x_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.96      0.94      8595
           1       0.58      0.43      0.49      1118

    accuracy                           0.90      9713
   macro avg       0.75      0.69      0.72      9713
weighted avg       0.89      0.90      0.89      9713



In [54]:
# Hyperparametertuning XGBOOST
param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

xgb_ft = RandomizedSearchCV(xgb, param_distributions=param_grid_xgb, n_iter=10, cv=5, random_state=42)
xgb_ft.fit(x_train_smote, y_train_smote)
xgb_ft.best_params_

{'subsample': 0.8,
 'n_estimators': 200,
 'max_depth': 7,
 'learning_rate': 0.3,
 'colsample_bytree': 1.0}

In [56]:
y_pred = xgb_ft.predict(x_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.94      0.94      8595
           1       0.53      0.51      0.52      1118

    accuracy                           0.89      9713
   macro avg       0.74      0.73      0.73      9713
weighted avg       0.89      0.89      0.89      9713

